In [0]:
from pyspark.sql import functions as F

tables = spark.sql("""SELECT DISTINCT TableName -- Include SourceName
                    FROM facttransaction.lcf.bronzetosilver
                    --WHERE TableName = 'POLICY'
                    """)
# tables.display()

for tablename in tables.collect():
    
    source_table_name = "facttransaction.bronze." + tablename.TableName
    print(source_table_name)
    target_table_name = "facttransaction.silver." + tablename.TableName
    
    actions = spark.sql(f"""SELECT SourceName, TableName, ColumnName, ActionName, ActionQuery
                        FROM facttransaction.lcf.bronzetosilver
                        JOIN facttransaction.lcf.action
                            ON action.ActionId = bronzetosilver.ActionId
                        WHERE TableName = '{tablename.TableName}'""")
    # actions.display()

    df = spark.read.table(source_table_name).drop("InsertDate").distinct()
    # df.display()

    for row in actions.collect():

        expression = None
        if row.ActionQuery is not None:
            expression = row.ActionQuery.replace("{ColumnName}", row.ColumnName)
            # print(expression)

        if row.ActionName == "All":
            for column_name, data_type in df.dtypes:
                if data_type == "string":
                    df = df.withColumn(column_name, F.regexp_replace(F.regexp_replace(F.trim(F.col(column_name)), r"\t", " "), r"\n", " "))
            
        else:
            df = df.withColumn(row.ColumnName, eval(expression))
             
    df = df.withColumn("InsertDate", F.current_timestamp())
    df.write.mode("append").saveAsTable(target_table_name)
    # df.display()
    df = None

In [0]:
%sql
SELECT CONCAT('UNION SELECT "', table_name, '", COUNT(*) FROM facttransaction.silver.', table_name) FROM facttransaction.information_schema.tables WHERE table_schema = 'silver';

SELECT "sourceb_policy_details", COUNT(*) FROM facttransaction.silver.sourceb_policy_details
UNION SELECT "sourceb_tax_master", COUNT(*) FROM facttransaction.silver.sourceb_tax_master
UNION SELECT "sourcea_premiums", COUNT(*) FROM facttransaction.silver.sourcea_premiums
UNION SELECT "sourceb_customers", COUNT(*) FROM facttransaction.silver.sourceb_customers
UNION SELECT "sourcea_ledger", COUNT(*) FROM facttransaction.silver.sourcea_ledger
UNION SELECT "sourcea_depts", COUNT(*) FROM facttransaction.silver.sourcea_depts
UNION SELECT "sourcea_tax", COUNT(*) FROM facttransaction.silver.sourcea_tax
UNION SELECT "sourcea_policymarket", COUNT(*) FROM facttransaction.silver.sourcea_policymarket
UNION SELECT "sourceb_premium_tax", COUNT(*) FROM facttransaction.silver.sourceb_premium_tax
UNION SELECT "sourcea_names", COUNT(*) FROM facttransaction.silver.sourcea_names
UNION SELECT "sourceb_department_master", COUNT(*) FROM facttransaction.silver.sourceb_department_master
UNION SELECT "sourcea_policy", COUNT(*) FROM facttransaction.silver.sourcea_policy
UNION SELECT "sourceb_gl_transactions", COUNT(*) FROM facttransaction.silver.sourceb_gl_transactions
UNION SELECT "sourceb_premium_header", COUNT(*) FROM facttransaction.silver.sourceb_premium_header
UNION SELECT "sourcea_taxcodes", COUNT(*) FROM facttransaction.silver.sourcea_taxcodes;

In [0]:
%sql
SELECT SourceName, TableName, ColumnName, ActionName, ActionQuery
FROM facttransaction.lcf.bronzetosilver
JOIN facttransaction.lcf.action
    ON action.ActionId = bronzetosilver.ActionId;

SELECT * FROM facttransaction.bronze.policy;
SELECT * FROM facttransaction.silver.depts;

-- UPDATE facttransaction.bronze.policy
-- SET PolicyNo = lower(PolicyNo)
--     ,CCY = lower(CCY);
